# La curva de precios a 20 años, paso a paso · TFM Energía UCM

Este notebook recorre el algoritmo de `scripts/curva_fundamental.py` **con una gráfica por
paso**, y termina en el mismo sitio que la sección 10 del notebook 07: la curva horaria de
2026 a 2046.

La idea del algoritmo cabe en una frase:

> En vez de suponer a cuánto estará el precio, se supone **cuánta capacidad habrá y a cuánto
> el gas** —cosas con objetivos publicados—, se reconstruye qué **demanda residual** dejaría
> eso bajo el tiempo que ya hizo, y se le pregunta al **merit order histórico** qué precio le
> corresponde.

El precio nunca entra. Siempre sale.

| paso | qué hace | figura |
|---|---|---|
| 1 | el panel de datos | cobertura y precio histórico |
| 2 | rendimiento solar y eólico | generación contra recurso |
| 3 | demanda residual | su desplome, año a año |
| 4 | elasticidad al gas | log-log por bin de residual |
| 5 | la curva de oferta | `k` y `P(precio ≤ 0)` |
| 6 | el sorteo meteorológico | un día futuro y su día molde |
| 7 | el suelo en cero | la masa puntual |
| 8 | el ruido por bloques | ACF y dispersión agregada |
| 9 | los 200 escenarios | una semana en spaghetti |
| 10 | **el resultado** | **la curva horaria, 20 años** |

In [ ]:
import sys, importlib
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "gold").is_dir())
sys.path.append(str(REPO / "scripts"))

import curva_fundamental as cfun, curva_precios
importlib.reload(cfun); importlib.reload(curva_precios)
from curva_precios import por_anclas, historico, curva

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": .3,
                     "axes.spines.top": False, "axes.spines.right": False})
ROJO, AZUL, GRIS = "#c0392b", "#2874a6", "#7f8c8d"

H = historico()
print(f"histórico de precio: {H.dia.min():%Y-%m-%d} -> {H.dia.max():%Y-%m-%d} "
      f"· {len(H):,} horas")

## Paso 1 · El panel

Todo sale de `matriz_produccion`, que se reconstruye a diario. Una fila por hora con precio,
demanda prevista, renovable prevista, gas, CO₂, radiación y viento — más la capacidad
instalada de solar y eólica, día a día.

**Es la matriz de producción y no el núcleo a propósito.** El núcleo está congelado para que
los escaladores sigan casando con los nueve modelos entrenados; si se moviera, dejarían de ser
válidos sin avisar.

In [ ]:
P = cfun.panel()
print(f"{len(P):,} horas · {P.ano.min()}-{P.ano.max()} · {P.dia.nunique():,} días")
print(f"columnas: {[c for c in P.columns if c not in ('dia','hora','ano')]}")

fig, ax = plt.subplots(2, 1, figsize=(13, 6), sharex=True,
                       gridspec_kw={"height_ratios": [2, 1]})
d = P.groupby("dia").agg(precio=("precio", "mean"), gas=("gas_mibgas", "mean"),
                         sol=("solar_gw", "mean"), eol=("eolica_gw", "mean"))
ax[0].plot(d.index, d.precio, lw=.4, color=GRIS, alpha=.8)
ax[0].plot(d.index, d.precio.rolling(30, center=True).mean(), lw=2, color="black",
           label="precio medio diario (media móvil 30 d)")
ax[0].plot(d.index, d.gas.rolling(30, center=True).mean(), lw=2, color=ROJO,
           label="gas MIBGAS")
ax[0].axhline(0, color="grey", lw=.8)
ax[0].set_ylabel("€/MWh"); ax[0].legend()
ax[0].set_title("Lo que entra: precio y combustible")

ax[1].fill_between(d.index, 0, d.sol, alpha=.6, color="#f39c12", label="solar")
ax[1].fill_between(d.index, d.sol, d.sol + d.eol, alpha=.6, color=AZUL, label="eólica")
ax[1].set_ylabel("GW instalados"); ax[1].legend(loc="upper left")
ax[1].set_title("Y la capacidad, que es lo que se proyecta al futuro")
plt.tight_layout(); plt.show()

## Paso 2 · El rendimiento: de meteorología a generación potencial

Cuánta generación da cada GW instalado por unidad de recurso.

```
solar  = η_s × radiación × GW_solar
eólica = η_e × viento³   × GW_eólica
```

**Se ajusta solo con las horas de precio > 5 €/MWh.** Ahí la planta produce todo lo que puede,
así que la generación observada mide *el recurso*. Si se metieran las horas baratas, el vertido
entraría dentro del rendimiento — y el vertido es consecuencia del precio, que es justo lo que
queremos predecir. Contarlo dos veces sesgaría la curva al alza.

In [ ]:
potencial, ir = cfun.rendimientos(P)
print(f"ajustado con {ir['horas_limpias']:,} horas de {ir['de']:,} "
      f"(precio > {cfun.PRECIO_LIMPIO:g} €/MWh)")
print(f"  η solar  = {ir['eta_solar']}   R² = {ir['R2_solar']}")
print(f"  η eólica = {ir['eta_eolica']}  R² = {ir['R2_eolica']}   <- flojo, ver nota")

q = P[P.precio > cfun.PRECIO_LIMPIO].sample(8000, random_state=42)
sol_p, eol_p = potencial(q.ssrd_meteo, q.wind100_meteo, q.solar_gw, q.eolica_gw)

fig, ax = plt.subplots(1, 2, figsize=(13, 4.4))
for a_, x, yv, nom, col in [(ax[0], sol_p, q.ree_gsolar_prev, "solar", "#f39c12"),
                            (ax[1], eol_p, q.ree_gwind_prev, "eólica", AZUL)]:
    a_.scatter(x, yv, s=4, alpha=.25, color=col)
    lim = [0, max(np.percentile(x, 99.5), np.percentile(yv, 99.5))]
    a_.plot(lim, lim, "--", color="black", lw=1.2, label="identidad")
    a_.set_xlabel(f"{nom} POTENCIAL estimada (MW)")
    a_.set_ylabel(f"{nom} observada (MW)")
    a_.set_title(f"{nom}: recurso -> generación")
    a_.legend()
plt.tight_layout(); plt.show()

print("\nEl viento se ajusta peor porque se usa el CUBO DE LA MEDIA espacial, y la media")
print("de los cubos no es el cubo de la media: promediar el viento sobre toda la península")
print("antes de elevarlo al cubo destruye la no linealidad. Se arregla con los tensores")
print("ECMWF a 0,25°, que dan la distribución espacial.")

## Paso 3 · La demanda residual

Lo que queda por cubrir con centrales térmicas:

```
residual = demanda − solar_potencial − eólica_potencial
```

**Esta es la variable del algoritmo.** No la capacidad solar instalada, que es lo que usaba la
versión anterior: la solar sola correla −0,52 con el precio horario y está confundida con la
eólica porque las dos crecen con el tiempo. La residual correla **+0,78 a +0,84** dentro de
cada año, y es la variable del *merit order*: física, no estadística.

In [ ]:
D = cfun.con_residual(P, potencial)

print(f"  {'año':>5s} {'residual media':>15s} {'precio':>8s} {'corr(precio,residual)':>22s}")
print("  " + "-" * 54)
for a, g in D.groupby("ano"):
    print(f"  {a:5d} {g.residual.mean():15,.0f} {g.precio.mean():8.1f} "
          f"{g.precio.corr(g.residual):22.3f}")

fig, ax = plt.subplots(1, 2, figsize=(13, 4.4))
for a, col in zip(sorted(D.ano.unique()),
                  plt.cm.viridis(np.linspace(0, .9, D.ano.nunique()))):
    ax[0].plot(D[D.ano == a].groupby("hora").residual.mean(), lw=2, color=col, label=a)
ax[0].set_xticks(range(0, 24, 3)); ax[0].set_xlabel("hora del día")
ax[0].set_ylabel("demanda residual (MW)")
ax[0].set_title("El valle de mediodía se hunde año a año")
ax[0].legend(fontsize=8, ncol=2)

sub = D.sample(9000, random_state=1)
sc = ax[1].scatter(sub.residual, sub.precio, s=4, c=sub.ano, cmap="viridis", alpha=.45)
ax[1].axhline(0, color="black", lw=.9)
ax[1].set_xlabel("demanda residual (MW)"); ax[1].set_ylabel("precio (€/MWh)")
ax[1].set_title("Y el precio la sigue")
plt.colorbar(sc, ax=ax[1], label="año")
plt.tight_layout(); plt.show()

## Paso 4 · La elasticidad al gas

El precio no es proporcional al gas. Lo sería si el ciclo combinado marginara siempre, y no
marginа siempre: compiten hidráulica e importaciones, y cada vez más horas las fija la
renovable.

```
precio ∝ gas^β
```

**β se estima con el histórico entero, no con la ventana reciente**, y el motivo es de
identificación: en 2023-2025 el gas estuvo en 38,7 / 34,4 / 35,9 €/MWh, prácticamente plano.
De ahí no se puede aprender cómo responde el precio al gas. En el histórico completo va de 4,2
a 225,0.

In [ ]:
beta = cfun.elasticidad_gas(D)
q = D[(D.precio > 1) & (D.gas_mibgas > 1)].copy()
q["b"] = pd.qcut(q.residual, 20, labels=False, duplicates="drop")
pend = q.groupby("b").apply(
    lambda g: np.polyfit(np.log(g.gas_mibgas), np.log(g.precio), 1)[0]
    if g.gas_mibgas.nunique() >= 50 else np.nan, include_groups=False).dropna()

fig, ax = plt.subplots(1, 2, figsize=(13, 4.4))
cen = q.groupby("b").residual.mean().loc[pend.index]
ax[0].plot(cen, pend.values, "o-", color=ROJO, lw=2)
ax[0].axhline(beta, ls="--", color="black", lw=1.4,
              label=f"β ponderado = {beta:.3f}")
ax[0].axhline(1.0, ls=":", color=GRIS, lw=1.4, label="proporcional (β = 1)")
ax[0].set_xlabel("demanda residual (MW)"); ax[0].set_ylabel("β del bin")
ax[0].set_title("β es estable entre bins, y siempre menor que 1")
ax[0].legend(fontsize=9)

m = q.sample(9000, random_state=3)
ax[1].scatter(m.gas_mibgas, m.precio, s=4, alpha=.2, color=AZUL)
g_ = np.linspace(q.gas_mibgas.min(), q.gas_mibgas.max(), 100)
esc = q.precio.median() / q.gas_mibgas.median() ** beta
ax[1].plot(g_, esc * g_ ** beta, color=ROJO, lw=2.2, label=f"gas^{beta:.2f}")
ax[1].plot(g_, (q.precio.median() / q.gas_mibgas.median()) * g_, "--", color="black",
           lw=1.4, label="proporcional")
ax[1].set_xscale("log"); ax[1].set_yscale("log")
ax[1].set_xlabel("gas MIBGAS (€/MWh)"); ax[1].set_ylabel("precio (€/MWh)")
ax[1].set_title("El gas varía de 4 a 225: ahí sí se puede estimar")
ax[1].legend(fontsize=9)
plt.tight_layout(); plt.show()
print(f"β = {beta:.3f}  ·  min por bin {pend.min():.2f} · max {pend.max():.2f}")

## Paso 5 · La curva de oferta

Aquí está el corazón. Se parte la demanda residual en 40 tramos y en cada uno se miden **dos**
cosas, con los últimos 3 años porque el merit order se mueve:

- **`k(residual)`** — la mediana de `precio / gas^β`. Es el *heat rate* implícito de la planta
  marginal: cuánto cuesta la hora por cada euro de gas. Se fuerza monótona creciente, que es lo
  mínimo que respeta el orden de mérito.
- **`P(precio ≤ 0)`** — la probabilidad de que la hora se case en el suelo. Sale monótona
  decreciente sola, no hay que imponerla.

Dividir por el gas **antes** de ajustar es lo que hace que la curva extrapole a veinte años: lo
que se memoriza es la forma del merit order, no el nivel de precios de 2020-2024.

In [ ]:
precio_of, ic = cfun.curva_oferta(D)
print(f"ajustada {ic['ajustada_desde']}-{ic['hasta']} · {ic['horas']:,} horas · "
      f"{ic['bins']} bins · β = {ic['beta_gas']}")
print(f"k de {ic['k_min']} a {ic['k_max']}  ·  P(≤0) de {ic['p0_max']} a {ic['p0_min']}")

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(ic["centro"], ic["k"], "o-", ms=4, color=ROJO, lw=2.4,
        label="k = precio / gas^β  (heat rate implícito)")
ax.set_xlabel("demanda residual (MW)")
ax.set_ylabel("k", color=ROJO); ax.tick_params(axis="y", labelcolor=ROJO)
ax.set_title("La curva de oferta, dibujada por los datos", fontsize=13)
b = ax.twinx(); b.grid(False)
b.plot(ic["centro"], ic["p0"], "s--", ms=4, color=AZUL, lw=2,
       label="P(precio ≤ 0)")
b.set_ylabel("P(precio ≤ 0)", color=AZUL); b.tick_params(axis="y", labelcolor=AZUL)
b.fill_between(ic["centro"], 0, ic["p0"], alpha=.12, color=AZUL)
h1, l1 = ax.get_legend_handles_labels(); h2, l2 = b.get_legend_handles_labels()
ax.legend(h1 + h2, l1 + l2, loc="center right", fontsize=9)
plt.tight_layout(); plt.show()

print("\n  A la izquierda margina la renovable y la hora tiende a cero.")
print("  A la derecha margina la térmica cara y la hora cuesta 3,5 veces el gas.")
print("  Entre medias está todo el mercado español de los próximos veinte años.")

## Paso 6 · El sorteo meteorológico

Para cada día del futuro se sortea **un día histórico del mismo día de calendario**, entre los
últimos 6 años, y se le roban su radiación y su viento hora a hora.

Días **enteros**, no horas sueltas: un mediodía soleado tiene que ir con su tarde soleada. Si
se rompiera esa correlación intradiaria la banda saldría absurdamente estrecha.

Luego ese tiempo pasado se evalúa con la **capacidad futura**. Ahí está el truco: la
meteorología es exógena —el precio no cambia cuánto sol hace— y la capacidad es el escenario
que aportas.

In [ ]:
# un 15 de julio cualquiera del futuro: qué días molde hay para sortear
MES, DIA = 7, 15
molde = cfun._meteo_molde(D)
cand = molde[(molde.mes == MES) & (molde.dm == DIA)]

fig, ax = plt.subplots(1, 2, figsize=(13, 4.4))
for dia_, g in cand.groupby("dia"):
    ax[0].plot(g.hora, g.ssrd_meteo, lw=1.6, alpha=.8, label=f"{dia_:%Y}")
    ax[1].plot(g.hora, g.wind100_meteo, lw=1.6, alpha=.8, label=f"{dia_:%Y}")
ax[0].set_xticks(range(0, 24, 3)); ax[0].set_xlabel("hora")
ax[0].set_ylabel("radiación"); ax[0].set_title(f"Los {cand.dia.nunique()} días molde del {DIA}-{MES:02d}")
ax[0].legend(fontsize=8, ncol=2)
ax[1].set_xticks(range(0, 24, 3)); ax[1].set_xlabel("hora")
ax[1].set_ylabel("viento a 100 m"); ax[1].set_title("Y su viento")
ax[1].legend(fontsize=8, ncol=2)
plt.tight_layout(); plt.show()

print(f"Cada escenario elige uno de esos {cand.dia.nunique()} días y lo evalúa con la")
print("capacidad del año que toque. Con 200 escenarios, la variabilidad de la banda es")
print("METEOROLÓGICA — la única que este modelo puede defender con datos.")
print("\nLímite honesto: seis años climáticos son seis sorteos. El sector usa treinta.")
print("Con seis, los años secos y sin viento están infrarrepresentados.")

## Paso 7 · El suelo en cero

Por cada hora se tira un dado:

```
si  u < P(precio ≤ 0 | residual):   la hora cae al suelo
                                    41 % exactamente 0,00
                                    el resto, de la cola negativa histórica
si no:                              precio = gas^β × k(residual) × ruido
```

Ese `if` es lo que **ninguna distribución continua puede hacer**. Y hace falta: las horas a
precio cero o negativo han pasado del 0,00 % en 2020 al 15,50 % en 2026, y un 3,82 % se casan
a **cero exacto**. El 31 de agosto de 2026 tuvo siete horas seguidas a 0,00.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.4))

hz = P.groupby("ano").precio.agg(cero=lambda x: (x == 0).mean() * 100,
                                 neg=lambda x: (x < 0).mean() * 100)
ax[0].bar(hz.index, hz.cero, color=ROJO, label="exactamente 0,00")
ax[0].bar(hz.index, hz.neg, bottom=hz.cero, color=AZUL, label="negativo")
ax[0].set_ylabel("% de horas del año"); ax[0].set_xlabel("año")
ax[0].set_title("El suelo del mercado, año a año"); ax[0].legend()

ult = P[P.ano == P.ano.max()]
ax[1].hist(ult.precio, bins=90, color=GRIS, alpha=.85)
ax[1].axvline(0, color=ROJO, lw=2)
n0 = (ult.precio == 0).sum()
ax[1].annotate(f"{n0:,} horas\nexactamente en 0", (0, ax[1].get_ylim()[1] * .75),
               xytext=(28, ax[1].get_ylim()[1] * .82), fontsize=9, color=ROJO,
               arrowprops=dict(arrowstyle="->", color=ROJO))
ax[1].set_xlabel("€/MWh"); ax[1].set_ylabel("horas")
ax[1].set_title(f"Distribución del precio en {P.ano.max()}")
plt.tight_layout(); plt.show()

print(f"  de los precios ≤ 0 del histórico, un {ic['frac_cero_exacto']:.0%} son cero exacto")

## Paso 8 · El ruido, por bloques y no independiente

Lo que la curva de oferta no explica se remuestrea del residuo real, **en bloques contiguos de
30 días**.

Y no es un capricho. La clase de series temporales exige diagnosticar la ACF del residuo antes
de dar los intervalos por buenos, y aquí el diagnóstico es demoledor: la ACF vale 0,88 a un
retardo y sigue en 0,14 al mes. **No es ruido blanco en ninguna escala.**

La consecuencia es que un ruido independiente se promedia al agregar y el real no. Con ruido
i.i.d. la banda anual de la curva a veinte años salía de **±1 €/MWh**.

### Y la dispersión no puede ser única

Hay un segundo problema, independiente del anterior. El residuo **no es homocedástico**: por
decil de demanda residual, su desviación va de **1,392 abajo a 0,174 arriba — ocho veces** — y
el sesgo pasa de −0,19 a −2,63. Físicamente es lo esperado: con el ciclo combinado marginando,
el precio está clavado al coste del combustible y no tiene dónde moverse; con la renovable
marginando, puede pasar cualquier cosa.

Con un σ único, la hora de pico recibía ocho veces la dispersión que le toca y, multiplicada
por una `k` de 10,6, producía precios de **653 €/MWh que no han existido nunca** (el máximo real
de 2025 fue 240). Y al valle le faltaba cola negativa.

Así que el ruido se monta como una **cópula empírica**: los bloques dan la dependencia
temporal, y el percentil resultante se mapea contra la distribución empírica **del tramo al que
pertenece cada hora simulada**. Dependencia por un lado, marginales por otro, sin postular
ninguna forma.

| | sesgo | curtosis | máximo |
|---|---|---|---|
| ruido blanco | 2,64 | 15,2 | 2.133 |
| bloques, marginal única | 2,17 | 12,5 | 653 |
| **bloques + marginal por tramo** | **−0,03** | **−1,1** | **246** |
| **real** | **0,15** | **−0,9** | **240** |

Encima va una **calibración asimétrica** de la banda —×2,00 por abajo, ×1,15 por arriba— porque
el residuo dentro de muestra subestima el error fuera de muestra, y lo hace más por abajo. Sube
la cobertura de 70,3 % a 80,5 %.

**Pero esa calibración no transfiere entre regímenes**, y conviene decirlo: 80,5 % en 2025, que
es el año con el que se midió, y 70,6 % en 2024, cuya ventana de ajuste incluye la crisis del
gas. Es un parche, no una ley. Lo que sí transfiere es la forma.

In [ ]:
res = ic["resid"]
lags = [1, 2, 3, 6, 12, 24, 48, 168, 336, 720]
acf = [float(np.corrcoef(res[:-L], res[L:])[0, 1]) for L in lags]

fig, ax = plt.subplots(1, 2, figsize=(13, 4.4))
ax[0].stem(lags, acf, basefmt=" ")
ax[0].axhline(2 / np.sqrt(len(res)), ls="--", color=ROJO, lw=1.2,
              label="banda de ruido blanco (±2/√n)")
ax[0].axhline(-2 / np.sqrt(len(res)), ls="--", color=ROJO, lw=1.2)
ax[0].set_xscale("log"); ax[0].set_xlabel("retardo (horas, escala log)")
ax[0].set_ylabel("ACF"); ax[0].set_title("El residuo tiene memoria a todas las escalas")
ax[0].legend(fontsize=9)

esc = {"hora": 1, "día": 24, "semana": 168, "mes": 720}
serie = pd.Series(res)
real_sd = [serie.std()] + [serie.groupby(serie.index // v).mean().std()
                           for v in list(esc.values())[1:]]
teor_sd = [serie.std() / np.sqrt(v) for v in esc.values()]
x = np.arange(len(esc))
ax[1].bar(x - .2, real_sd, .4, color=ROJO, label="dispersión real")
ax[1].bar(x + .2, teor_sd, .4, color=AZUL, label="si fuera ruido blanco")
ax[1].set_xticks(x); ax[1].set_xticklabels(esc.keys()); ax[1].set_yscale("log")
ax[1].set_ylabel("sd del residuo agregado"); ax[1].legend()
ax[1].set_title("Por eso la banda salía estrecha")
plt.tight_layout(); plt.show()

for k_, r_, t_ in zip(esc, real_sd, teor_sd):
    print(f"  por {k_:7s} real {r_:.3f}   ruido blanco {t_:.3f}   x{r_/t_:.1f}")
print("\nUn AR(1) tampoco bastaría: con φ=0,88 decae a 0,88^720 ≈ 0 en un mes, y la")
print("persistencia mensual medida es 0,14. Los bloques conservan todas las escalas.")

## Paso 9 · Los escenarios

Ya está todo. Se aportan los **cuatro escenarios anuales** —gas, demanda, GW solares, GW
eólicos— y se repite el sorteo 200 veces.

Ninguno de los cuatro es un precio. El primer valor de cada uno es **el observado hoy**, así
que la curva empalma con la realidad en vez de arrancar de un número inventado.

In [ ]:
ULT_REAL = H.dia.max()
SIM_DESDE = ULT_REAL + pd.Timedelta(days=1)
A_SIM, ANO_FIN = SIM_DESDE.year, 2046

obs = P[P.ano == P.ano.max()]
GAS_HOY, DEM_HOY = float(obs.gas_mibgas.mean()), float(obs.demanda.mean())
SOL_HOY, EOL_HOY = float(obs.solar_gw.mean()), float(obs.eolica_gw.mean())

ESC = dict(
    gas=por_anclas({A_SIM: GAS_HOY, 2035: GAS_HOY * .82, ANO_FIN: GAS_HOY * .74},
                   A_SIM, ANO_FIN),
    demanda=por_anclas({A_SIM: DEM_HOY, ANO_FIN: DEM_HOY * 1.01 ** (ANO_FIN - A_SIM)},
                       A_SIM, ANO_FIN),
    solar_gw=por_anclas({A_SIM: SOL_HOY, 2030: 76, 2035: 95, 2040: 110, ANO_FIN: 125},
                        A_SIM, ANO_FIN),
    eolica_gw=por_anclas({A_SIM: EOL_HOY, 2030: 43, 2040: 55, ANO_FIN: 62},
                         A_SIM, ANO_FIN))

print(f"simulando {SIM_DESDE:%Y-%m-%d} -> {ANO_FIN}-12-31  "
      f"({ANO_FIN - A_SIM + 1} años de calendario)")
print(f"  {'año':>5s} {'gas':>7s} {'demanda':>9s} {'solar GW':>9s} {'eólica GW':>10s}")
for a in [A_SIM, 2030, 2035, 2040, ANO_FIN]:
    print(f"  {a:5d} {ESC['gas'][a]:7.1f} {ESC['demanda'][a]:9,.0f} "
          f"{ESC['solar_gw'][a]:9.0f} {ESC['eolica_gw'][a]:10.0f}")

CF, SIMS = cfun.simular(SIM_DESDE, f"{ANO_FIN}-12-31", **ESC, potencial=potencial,
                        precio=precio_of, n=200, verbose=False, crudo=True)
print(f"\n{SIMS.shape[0]} escenarios x {SIMS.shape[1]:,} horas = "
       f"{SIMS.size:,} precios simulados")

# una semana concreta, escenario a escenario
SEM = f"{(A_SIM + ANO_FIN) // 2}-07-06"
m = (CF.dia >= SEM) & (CF.dia < pd.Timestamp(SEM) + pd.Timedelta(days=7))
ts = (CF.dia + pd.to_timedelta(CF.hora, unit="h"))[m]
fig, ax = plt.subplots(figsize=(13, 4.6))
for k in range(0, 200, 4):
    ax.plot(ts, SIMS[k][m.to_numpy()], lw=.5, color=GRIS, alpha=.35)
ax.plot(ts, CF.p50[m], color=ROJO, lw=2.4, label="P50")
ax.fill_between(ts, CF.p10[m], CF.p90[m], alpha=.2, color=ROJO, label="P10-P90")
ax.axhline(0, color="black", lw=.9)
ax.set_ylabel("€/MWh"); ax.legend()
ax.set_title(f"50 de los 200 escenarios · semana del {pd.Timestamp(SEM):%d-%m-%Y}")
plt.tight_layout(); plt.show()

## Paso 10 · El resultado: la curva horaria de los 20 años

Tres vistas del mismo objeto.

**El mapa** es la más informativa: 24 horas en el eje x, un año por fila. Se lee de arriba
abajo cómo el valle de mediodía se hunde hasta el cero mientras el pico de la tarde aguanta.

In [ ]:
y = CF.dia.dt.year.to_numpy()
mapa = CF.assign(a=y).pivot_table(index="a", columns="hora", values="p50", aggfunc="mean")

fig, ax = plt.subplots(1, 2, figsize=(14, 6.5),
                       gridspec_kw={"width_ratios": [1.25, 1]})
norm = TwoSlopeNorm(vmin=min(mapa.values.min(), -1), vcenter=0, vmax=mapa.values.max())
im = ax[0].pcolormesh(mapa.columns, mapa.index, mapa.values, cmap="RdYlBu_r",
                      norm=norm, shading="nearest")
ax[0].set_xticks(range(0, 24, 2)); ax[0].set_xlabel("hora del día")
ax[0].set_ylabel("año"); ax[0].invert_yaxis(); ax[0].grid(False)
ax[0].set_title(f"Curva horaria P50 · {A_SIM}-{ANO_FIN}", fontsize=12)
plt.colorbar(im, ax=ax[0], label="€/MWh")

cols = plt.cm.plasma(np.linspace(.08, .92, len(mapa)))
for (a, fila), col in zip(mapa.iterrows(), cols):
    ax[1].plot(fila.index, fila.values, lw=1.6, color=col)
hoy = H[H.dia > H.dia.max() - pd.DateOffset(years=1)].groupby("hora").precio.mean()
ax[1].plot(hoy.index, hoy.values, "--", color="black", lw=2.4, label="hoy (12 m reales)")
ax[1].axhline(0, color="grey", lw=.9)
ax[1].set_xticks(range(0, 24, 3)); ax[1].set_xlabel("hora del día")
ax[1].set_ylabel("€/MWh"); ax[1].legend(fontsize=9)
ax[1].set_title(f"Los {len(mapa)} perfiles, de {A_SIM} (claro) a {ANO_FIN} (oscuro)",
                fontsize=12)
plt.tight_layout(); plt.show()

**La serie completa**, día a día, con el histórico cosido delante. Es el fichero que se
entrega: `curva()` une los tres orígenes y marca cada fila con el suyo.

In [ ]:
dia = CF.groupby("dia").agg(p10=("p10", "mean"), p50=("p50", "mean"),
                            p90=("p90", "mean"))
hd = H.groupby("dia").precio.mean()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(hd.index, hd, lw=.3, color=GRIS, alpha=.7)
ax.plot(hd.index, hd.rolling(90, center=True).mean(), lw=2.2, color="black",
        label="histórico")
ax.fill_between(dia.index, dia.p10, dia.p90, alpha=.16, color=ROJO, lw=0,
                label="simulado P10-P90")
ax.plot(dia.index, dia.p50, lw=.3, color=ROJO, alpha=.6)
ax.plot(dia.index, dia.p50.rolling(90, center=True).mean(), lw=2.2, color=ROJO,
        label="simulado P50 (media móvil 90 d)")
ax.axvline(SIM_DESDE, ls="--", color="black", lw=1.2)
ax.annotate("último precio\npublicado", (SIM_DESDE, ax.get_ylim()[1] * .88),
            xytext=(-95, 0), textcoords="offset points", fontsize=9,
            arrowprops=dict(arrowstyle="->"))
ax.axhline(0, color="grey", lw=.9)
ax.set_ylabel("€/MWh"); ax.legend(loc="upper right")
ax.set_title(f"{H.dia.min():%Y} - {ANO_FIN} · media diaria · "
             f"{len(H) + len(CF):,} horas en total", fontsize=13)
plt.tight_layout(); plt.show()

**Y la tabla**, que es lo que se pega en la memoria.

Un aviso para leerla: **la fila del primer año no es un año completo**. Empieza el día
siguiente al último precio publicado, así que solo cubre los meses que quedan — otoño e
invierno, que son los caros. Por eso sale por encima del año siguiente. No es una
discontinuidad del modelo, es que no son doce meses.

In [ ]:
anf = CF.assign(a=y).groupby("a")[["p10", "p50", "p90"]].mean()
anf["media"] = [SIMS[:, y == k].mean() for k in anf.index]
anf["h_cero_%"] = [(SIMS[:, y == k] <= 0).mean() * 100 for k in anf.index]
anf["solar_GW"] = pd.Series(ESC["solar_gw"])
anf["gas"] = pd.Series(ESC["gas"])
rel = CF.p50 - CF.groupby("dia").p50.transform("mean")
anf["spread"] = (rel[CF.hora.between(19, 21)].groupby(y[CF.hora.between(19, 21)]).mean()
                 - rel[CF.hora.between(12, 15)].groupby(y[CF.hora.between(12, 15)]).mean())
display(anf.round(1))

print(f"\n  CURVA HORARIA CONSOLIDADA {A_SIM}-{ANO_FIN}")
cons = CF.groupby("hora").agg(minimo=("p10", "mean"), estimacion=("p50", "mean"),
                              maximo=("p90", "mean")).round(2)
print(f"  {'hora':>5s} {'mínimo':>9s} {'estimación':>12s} {'máximo':>9s} {'banda':>8s}")
print("  " + "-" * 50)
for h, r in cons.iterrows():
    marca = ("  <- valle" if h == cons.estimacion.idxmin() else
             ("  <- pico" if h == cons.estimacion.idxmax() else ""))
    print(f"  {h:5d} {r.minimo:9.2f} {r.estimacion:12.2f} {r.maximo:9.2f} "
          f"{r.maximo - r.minimo:8.1f}{marca}")

---

## Lo que hay que decir junto al resultado

**El nivel ya no se supone, pero el gas sí.** `k` multiplica directamente por `gas^0,713`. Se
ha ganado en que el supuesto es sobre una magnitud que cotiza a plazo y se puede contrastar, no
sobre el propio resultado — pero sigue siendo el supuesto que más pesa. La sección 10 del
notebook 07 mide cuánto: con gas plano el spread sube, con gas a la baja se estrecha.

**No hay realimentación de almacenamiento.** Hoy España tiene 235 MW de batería, así que el
efecto no se puede estimar de los datos: no hay variación. El PNIEC apunta a decenas de GW y el
almacenamiento vive de arbitrar el spread, de modo que **el spread de 2046 es un techo, no una
estimación**.

**La cobertura sigue corta.** 73-78 % en la banda P10-P90 frente al 80 objetivo. Tras el
remuestreo por bloques la *dependencia* ya está bien; lo que falta calibrar es la distribución
marginal de cada hora.

**Seis años climáticos son seis sorteos.** El sector usa treinta. La banda es más estrecha de
lo que debería por construcción, y eso no se arregla con estadística: se arregla descargando
más ERA5.